---
title: Crossover analysis
description: Using xOPR to automatically find radar crossovers
date: 2025-09-09
---

Radar sounder crossover analysis refers to finding crossing points between radar flight lines and analyzing differences between the radar data at the same point. Differences can arise from many factors. Often, crossing flight paths are at (nearly) perpendicular angles. Because the along-track and across-track beamwidth of most radar systems is very different, this different imaging geometry can lead to differences in the radar data. These differences are most pronouned over rough terrain, where off-nadir clutter may show up differently in coicident data collected along different angles. Temporal changes, changes in radar systems, and errors in picking the location of the surface or bed are other sources of differences in crossover analysis.

In this notebook, we demonstratate how to automatically find and analyze radar crossovers. To do this, we will

1. Find STAC items representing radar data within a geographic region.
2. Use the STAC item geometry to identify points where the radar flight paths cross (crossover points).
3. Selectively load layer information to make a map of the differences in WGS84 elevation between the bed picks at each crossover point. (This step is parallelized using Dask.)
4. Interactively load and plot radar data round selected crossover points to see what's happening.

In [ ]:
%load_ext autoreload
%autoreload 2

import numpy as np
import xarray as xr
import geoviews as gv
import geoviews.feature as gf
import cartopy.crs as ccrs
import matplotlib.pyplot as plt
import geopandas as gpd

import xopr

import holoviews as hv
import hvplot.xarray
import hvplot.pandas
hvplot.extension('bokeh')


In [ ]:
# Useful background features for maps
background_map = gf.ocean.opts(projection=ccrs.SouthPolarStereo(), scale='50m') * gf.coastline.opts(projection=ccrs.SouthPolarStereo(), scale='50m')

In [ ]:
# Establish an OPR session
# You'll probably want to set a cache directory if you're running this locally to speed
# up subsequent requests. You can do other things like customize the STAC API endpoint,
# but you shouldn't need to do that for most use cases.
opr = xopr.OPRConnection(cache_dir="radar_cache")

# Or you can open a connection without a cache directory (for example, if you're parallelizing
# this on a cloud cluster without persistent storage).
#opr = xopr.OPRConnection()

### Step 1: Finding radar lines

For this notebook, we'll experiment with finding data by geographic region. xOPR includes a helper module `xopr.geometry` with some useful utilities. You can call `xopr.geometry.get_antarctic_regions` to select one or more regions from the [MEaSUREs Antarctic Boundaries](https://nsidc.org/data/nsidc-0709/versions/2) dataset. Below, we plot all of the regions to give you some options for what to try. Mouse over each region to see its name.

In [ ]:
regions_df = xopr.geometry.get_antarctic_regions(merge_regions=False).to_crs('EPSG:3031')
regions_df.hvplot(frame_width=600, aspect='equal', hover_cols=['NAME'], c='TYPE')

For our demonstration, we'll use David Glacier, but feel free to trade this out for other locations and experiment.

In [ ]:
region = xopr.geometry.get_antarctic_regions(name="David", merge_regions=True, simplify_tolerance=100)
region_projected = xopr.geometry.project_geojson(region, source_crs='EPSG:4326', target_crs="EPSG:3031")

region_hv = hv.Polygons([region_projected]).opts(
    color='green',
    line_color='black',
    fill_alpha=0.5)

(background_map * region_hv).opts(aspect='equal')

In [ ]:
stac_items_df = opr.query_frames(geometry=region, date_range="2021-01-01T00:00:00Z/2025-06-01T00:00:00Z") # Can also add a date range: date_range="2020-01-01T00:00:00Z/2024-01-01T00:00:00Z"
stac_items_df = stac_items_df.to_crs('EPSG:3031')

print(f"Found {len(stac_items_df)} frames across {stac_items_df['collection'].nunique()} collections:")
stac_items_df.groupby('collection').size()

In [ ]:
flight_lines = stac_items_df.hvplot(by='collection')
(background_map * region_hv * flight_lines).opts(frame_width=500, aspect='equal', active_tools=['pan', 'wheel_zoom'])

### Step 2: Identify crossover points

xOPR includes a helper function to identify crossover points. Feel free to poke into the code and take a look. It takes advantage of GeoPandas's spatial join (`sjoin`) function, so it's actually quite concise.

```{tip} Note about runtimes
This notebook can take quite some time to run if you have a lot of crossover points. The bottleneck is loading layer picks (one network round-trip per frame); the per-pair computation is microseconds. If you're here to learn the library, we recommend limiting your queries to 100 or so intersection points. Some ways to do that:
- Choose a smaller geographic region or limit the date range of your query (above)
- Filter the intersections dataframe by the crossing angle -- this will filter out nearly coincident lines which produce a lot of crossovers: `intersections = intersections[intersections['crossing_angle'] > 45]`
- Just artificially limit the number of crossovers: `intersections = intersections.iloc[:100]`
```


In [ ]:
intersections = xopr.find_intersections(stac_items_df, calculate_crossing_angles=True)

intersections = intersections[intersections['crossing_angle'] > 2] # Filter out nearly coincident crossings

print(f"Found {len(intersections)} crossover points between flight lines.")
(background_map * region_hv * flight_lines * intersections.hvplot(label='Intersection Points', c='crossing_angle')).opts(frame_width=500, aspect='equal', active_tools=['pan', 'wheel_zoom'])

Zoom in and check out the plot above. Every identified crossover point has a blue dot over it. The blue dots are shaded by the crossing angle. You'll see that nearly coincident lines will have a large number of low crossing angles wheras most other crossovers are at angles closer to 90 degrees.

The cell below shows you what the result of `xopr.find_intersections()` looks like. It returns a GeoDataFrame where every column from the intersecting frames is preserved, with suffixes _1 and _2 to distinguish them.

In [ ]:
intersections.head()

### Step 3: Load layer data and find the difference in bed elevation

[`OPRConnection.load_bed_picks`](../api/xopr.opr_access.html#xopr.opr_access.OPRConnection.load_bed_picks) loads bed picks for every frame in one call, returning a flat GeoDataFrame tagged with the source frame's STAC id. We then iterate over the intersection points, slice the picks for each frame, and call [`xopr.compute_crossover_error`](../api/xopr.opr_tools.html#xopr.opr_tools.compute_crossover_error) to get the elevation at each frame's nearest pick to the crossover.

:::{tip} Note about runtimes
Loading picks is one network round-trip per frame, so this can take a while for large catalogs. The picks-vs-picks computation is fast (microseconds per pair), so the total time is dominated by the network step. If you're experimenting, limit your initial query to ~100 intersection points.
:::


In [ ]:
# Subset frames to those touching any intersection — saves loading picks for frames
# that don't participate in any crossover (mbox-based pre-filter when available).
frame_ids = set(intersections['id_1']).union(intersections['id_2'])
frames_used = stac_items_df.loc[stac_items_df['id'].isin(frame_ids)]
print(f"Loading bed picks for {len(frames_used)} frames participating in {len(intersections)} crossovers")

picks = opr.load_bed_picks(frames_used, target_crs='EPSG:3031')
print(f"Loaded {len(picks):,} bed picks")


In [ ]:
# Index picks by frame id for O(1) lookups in the loop below
picks_by_id = {fid: g for fid, g in picks.groupby('id')}


In [ ]:
from xopr.opr_tools import compute_crossover_error

# Project intersections to EPSG:3031 to match the picks
intersections_proj = intersections.to_crs('EPSG:3031')

for idx, row in intersections_proj.iterrows():
    p1 = picks_by_id.get(row['id_1'])
    p2 = picks_by_id.get(row['id_2'])
    if p1 is None or p2 is None or len(p1) == 0 or len(p2) == 0:
        continue
    elev_1, elev_2, dist = compute_crossover_error(p1, p2, row.intersection_geometry)
    intersections.at[idx, 'wgs84_1'] = elev_1
    intersections.at[idx, 'wgs84_2'] = elev_2
    intersections.at[idx, 'layer_pt_distance'] = dist


In [ ]:
intersections['elev_diff'] = np.abs(intersections['wgs84_1'] - intersections['wgs84_2'])
# Set elev_diff to NaN where layer_pt_distance is large
intersections.loc[intersections['layer_pt_distance'] > 100, 'elev_diff'] = np.nan

In [ ]:
intersections_success = intersections.dropna(subset=['wgs84_1', 'wgs84_2', 'elev_diff', 'layer_pt_distance']).reset_index(drop=True)

# Report how many intersections had valid layer data
n_total = len(intersections)
n_success = len(intersections_success)
print(f"Layer data available for {n_success}/{n_total} intersections ({100*n_success/n_total:.0f}%)")

if len(intersections_success) == 0:
    raise ValueError("No intersections with valid layer data found. Try expanding the date range or choosing a different region.")

intersections_success['intersection_geometry_x'] = intersections_success['intersection_geometry'].apply(lambda geom: geom.x)
intersections_success = intersections_success.sort_values(by='intersection_geometry_x', ascending=False).reset_index(drop=True)
intersections_success['idx'] = intersections_success.index
hover_tooltips = [
    ("Index", "@idx"),
    ("Collection 1", "@collection_1"),
    ("Collection 2", "@collection_2"),
    ("Difference", "@elev_diff{0.00} m"),
]

vlim = intersections_success['elev_diff'].abs().quantile(0.9)

hv_int = intersections_success.hvplot(color='elev_diff', hover_cols=['idx', 'collection_1', 'collection_2', 'elev_diff'], hover_tooltips=hover_tooltips, clim=(0, vlim))
#hv_int = hv_int.opts(scalebar=True) # Can enable if you want - requires hvplot >= 0.12.0
(background_map * region_hv * flight_lines * hv_int).opts(frame_width=600, aspect='equal', active_tools=['pan', 'wheel_zoom'])

This map shows the differences in picked bed elevation at every crossing point where layer data was available. Explore around and see what you notice!

### Step 4: Investigate individual crossovers by looking at the radar data

If you hover over any crossover point in the map above, you'll get the index associated with each crossover. Enter one of those indices below to load and plot the corresponding radar data to see what's happening.

In [ ]:
# Select an index to investigate - pick one from the map above, or use a default
# We default to the intersection with the largest elevation difference
print(f"Valid index range: 0 to {len(intersections_success)-1}")
selected_idx = intersections_success['elev_diff'].argmax()
print(f"Selected index: {selected_idx}")

In [ ]:
# Get intersection details
intersect = intersections_success.loc[selected_idx]
stac_1 = stac_items_df.loc[intersect['id_1']].to_dict()
stac_2 = stac_items_df.loc[intersect['id_2']].to_dict()

# Load frames
frame_1 = opr.load_frame(stac_1)
frame_1 = xopr.radar_util.add_along_track(frame_1)
frame_1 = xopr.radar_util.interpolate_to_vertical_grid(frame_1, vertical_coordinate='wgs84')
frame_2 = opr.load_frame(stac_2)
frame_2 = xopr.radar_util.add_along_track(frame_2)
frame_2 = xopr.radar_util.interpolate_to_vertical_grid(frame_2, vertical_coordinate='wgs84')

# Project to EPSG:3031 and find closest points to intersection
x_int, y_int = intersect.intersection_geometry.coords[0]
frame_1_proj = xopr.geometry.project_dataset(frame_1, "EPSG:3031")
frame_2_proj = xopr.geometry.project_dataset(frame_2, "EPSG:3031")

# Find indices closest to intersection
dist_1 = np.sqrt((frame_1_proj['x'] - x_int)**2 + (frame_1_proj['y'] - y_int)**2)
dist_2 = np.sqrt((frame_2_proj['x'] - x_int)**2 + (frame_2_proj['y'] - y_int)**2)
idx_1 = np.argmin(dist_1.data)
idx_2 = np.argmin(dist_2.data)

print(f"Frame 1: {intersect['id_1']} from {intersect['collection_1']}")
print(f"Frame 2: {intersect['id_2']} from {intersect['collection_2']}")
print(f"Intersection at index {idx_1} (frame 1) and {idx_2} (frame 2)")
print(f"Bed elevation difference: {intersect['elev_diff']:.2f} m")

In [ ]:
layers_1 = opr.get_layers(frame_1)
layers_2 = opr.get_layers(frame_2)

# Check for empty layers and report them (helps diagnose data issues)
for layer_idx in list(layers_1.keys()):
    if layers_1[layer_idx].sizes.get('slow_time', 0) == 0:
        print(f"WARNING: Empty layer '{layer_idx}' in frame_1 ({intersect['id_1']} from {intersect['collection_1']})")
        print(f"  This may indicate a time range mismatch between the frame and layer data")
        del layers_1[layer_idx]

for layer_idx in list(layers_2.keys()):
    if layers_2[layer_idx].sizes.get('slow_time', 0) == 0:
        print(f"WARNING: Empty layer '{layer_idx}' in frame_2 ({intersect['id_2']} from {intersect['collection_2']})")
        print(f"  This may indicate a time range mismatch between the frame and layer data")
        del layers_2[layer_idx]

# Process remaining layers
for layer_idx in layers_1:
    layers_1[layer_idx] = xopr.radar_util.add_along_track(layers_1[layer_idx])
    layers_1[layer_idx] = xopr.layer_twtt_to_range(layers_1[layer_idx], layers_1["standard:surface"], vertical_coordinate='wgs84')
    layers_1[layer_idx] = xopr.layer_twtt_to_range(layers_1[layer_idx], layers_1["standard:surface"], vertical_coordinate='range')

for layer_idx in layers_2:
    layers_2[layer_idx] = xopr.radar_util.add_along_track(layers_2[layer_idx])
    layers_2[layer_idx] = xopr.layer_twtt_to_range(layers_2[layer_idx], layers_2["standard:surface"], vertical_coordinate='wgs84')
    layers_2[layer_idx] = xopr.layer_twtt_to_range(layers_2[layer_idx], layers_2["standard:surface"], vertical_coordinate='range')

In [ ]:
clb_min_pct, clb_max_pct = 30, 97

# Plot radargrams in elevation coordinates with layers
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(15, 8))

# Frame 1 radargram in elevation
pwr_1_elev = 10*np.log10(np.abs(frame_1.Data))
vmax_1 = np.percentile(pwr_1_elev, clb_max_pct)
vmin_1 = np.percentile(pwr_1_elev, clb_min_pct)
pwr_1_elev.plot.imshow(x='along_track', y='wgs84', cmap='gray', ax=ax1, vmin=vmin_1, vmax=vmax_1)
ax1.axvline(frame_1.along_track[idx_1].values, color='red', linestyle='--', linewidth=2, label='Crossover')

# Plot layers using elevation data
for layer_name in layers_1:
    layers_1[layer_name]['wgs84'].plot(ax=ax1, x='along_track', linewidth=1, linestyle=':', label=layer_name)

ax1.set_title(f"{intersect['collection_1']} - {intersect['id_1']} (Elevation view)")
ax1.set_ylabel('Elevation (m)')
ax1.legend()

# Frame 2 radargram in elevation
pwr_2_elev = 10*np.log10(np.abs(frame_2.Data))
vmax_2 = np.percentile(pwr_2_elev, clb_max_pct)
vmin_2 = np.percentile(pwr_2_elev, clb_min_pct)
pwr_2_elev.plot.imshow(x='along_track', y='wgs84', cmap='gray', ax=ax2, vmin=vmin_2, vmax=vmax_2)
ax2.axvline(frame_2.along_track[idx_2].values, color='red', linestyle='--', linewidth=2, label='Crossover')

# Plot layers using elevation data
for layer_name in layers_2:
    layers_2[layer_name]['wgs84'].plot(ax=ax2, x='along_track', linewidth=1, linestyle=':', label=layer_name)

ax2.set_title(f"{intersect['collection_2']} - {intersect['id_2']} (Elevation view)")
ax2.set_xlabel('Along track distance (m)')
ax2.set_ylabel('Elevation (m)')
ax2.legend()

plt.suptitle(f"Radargrams in elevation coordinates - Bed elev diff: {intersect['elev_diff']:.2f} m", fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Zoomed elevation plots around crossover
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(15, 8))

window_size = 300  # Number of traces on each side of intersection
elev_window = np.maximum(intersect['elev_diff']*2.5, 100)  # Elevation window in meters around bed

# Frame 1 zoomed
idx_start_1 = max(0, idx_1 - window_size)
idx_end_1 = min(len(frame_1.slow_time), idx_1 + window_size)

# Convert indices to slow_time values for layer slicing
slow_time_start_1 = frame_1.slow_time[idx_start_1]
slow_time_end_1 = frame_1.slow_time[idx_end_1 - 1]

pwr_1_zoom = frame_1.Data.isel(slow_time=slice(idx_start_1, idx_end_1))
pwr_1_zoom_db = 10*np.log10(np.abs(pwr_1_zoom))
pwr_1_zoom_db.plot.imshow(x='along_track', y='wgs84', cmap='gray', ax=ax1, vmin=vmin_1, vmax=vmax_1)
ax1.axvline(frame_1.along_track[idx_1].values, color='red', linestyle='--', linewidth=2, label='Crossover')

# Plot layers using the layers_1 dictionary
for layer_name in layers_1:
    layer_zoom = layers_1[layer_name]['wgs84'].sel(slow_time=slice(slow_time_start_1, slow_time_end_1))
    layer_zoom.plot(ax=ax1, x='along_track', linewidth=1, linestyle=':', label=layer_name)

# Set y-limits based on bed layer if available
if 'standard:bottom' in layers_1:
    bed_elev_1 = layers_1['standard:bottom']['wgs84'].isel(slow_time=idx_1).values
    elev_min_1 = bed_elev_1 - elev_window/2
    elev_max_1 = bed_elev_1 + elev_window/2
    ax1.set_ylim(elev_min_1, elev_max_1)

ax1.set_title(f"{intersect['collection_1']} - Zoomed (Bed: {intersect['wgs84_1']:.1f} m)")
ax1.set_ylabel('Elevation (m)')
ax1.legend()

# Frame 2 zoomed
idx_start_2 = max(0, idx_2 - window_size)
idx_end_2 = min(len(frame_2.slow_time), idx_2 + window_size)

# Convert indices to slow_time values for layer slicing
slow_time_start_2 = frame_2.slow_time[idx_start_2]
slow_time_end_2 = frame_2.slow_time[idx_end_2 - 1]

pwr_2_zoom = frame_2.Data.isel(slow_time=slice(idx_start_2, idx_end_2))
pwr_2_zoom_db = 10*np.log10(np.abs(pwr_2_zoom))
pwr_2_zoom_db.plot.imshow(x='along_track', y='wgs84', cmap='gray', ax=ax2, vmin=vmin_2, vmax=vmax_2)
ax2.axvline(frame_2.along_track[idx_2].values, color='red', linestyle='--', linewidth=2, label='Crossover')

# Plot layers using the layers_2 dictionary
for layer_name in layers_2:
    layer_zoom = layers_2[layer_name]['wgs84'].sel(slow_time=slice(slow_time_start_2, slow_time_end_2))
    layer_zoom.plot(ax=ax2, x='along_track', linewidth=1, linestyle=':', label=layer_name)

# Set y-limits based on bed layer if available
if 'standard:bottom' in layers_2:
    bed_elev_2 = layers_2['standard:bottom']['wgs84'].isel(slow_time=idx_2).values
    elev_min_2 = bed_elev_2 - elev_window/2
    elev_max_2 = bed_elev_2 + elev_window/2
    ax2.set_ylim(elev_min_2, elev_max_2)

ax2.set_title(f"{intersect['collection_2']} - Zoomed (Bed: {intersect['wgs84_2']:.1f} m)")
ax2.set_xlabel('Along track distance (m)')
ax2.set_ylabel('Elevation (m)')
ax2.legend()

plt.suptitle(f"Elevation crossover comparison - Bed difference: {intersect['elev_diff']:.2f} m", fontsize=14)
plt.tight_layout()
plt.show()